# Stronger visual target, tri-modal alignment, and cross-subject calibration
Runtime: L4 or A100 GPU (results below were run on an L4). Uses the preprocessed subjects cached on Google Drive by `02_loso.ipynb`.

1. Captions (BLIP-large) and features for CLIP ViT-B/32 and ViT-H/14
2. Within-subject, 10 subjects: target size, EEG encoder (conv vs channel-attention transformer + conv), and EEG-image-text alignment
3. Leave-one-subject-out with ViT-H/14 for both encoders, then adaptation to the new person: AdaBN (no labels), subject-specific channel adapter and fine-tuning (250/1000/4000 labelled images)

4. Test-time refinement of retrieval scores (centering, CSLS), no labels
5. Within-subject refinement for all 10 subjects; ATM training recipe (single trials, batch 1024)

Every finished run is saved on Drive and skipped on re-run.
Reported run: the first three entries of `CONFIGS` and the first three entries of `X_FOLDS` (sub-01, sub-05, sub-08).

## Setup

In [ ]:
# Setup: mount Drive, paths, settings
import os, sys, gc, json, math, time, random, shutil, importlib.util, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
DRIVE  = os.environ.get("EEG_DRIVE", "/content/drive/MyDrive/eeg_results")   # results and checkpoints
DCACHE = f"{DRIVE}/cache"                                                     # preprocessed EEG and image features
ROOT   = os.environ.get("EEG_ROOT", "/content/EEG_Data")                      # raw downloads
LOCAL  = os.environ.get("EEG_LOCAL", "/content/cache")                        # local copy of the cache
for d in [DRIVE, DCACHE, ROOT, LOCAL]: os.makedirs(d, exist_ok=True)
ALL_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
CLIP_ARCH, CLIP_PRETRAINED = "ViT-B-32", "openai"
LR, WD = 3e-4, 1e-4

free_gb = shutil.disk_usage(DRIVE).free / 1e9
have = sum(os.path.exists(f"{DCACHE}/{s}_train_z16.npy") for s in ALL_SUBJECTS)
print(f"Drive free: {free_gb:.1f} GB | subjects already saved in Drive: {have}/10")
if free_gb < 0.55 * (10 - have) + 0.2:
    print("warning: low Drive space (about 0.53 GB per subject is needed)")

import numpy as np
for pkg, pip_name in [("open_clip", "open_clip_torch"), ("huggingface_hub", "huggingface_hub"), ("pandas", "pandas")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "", "| torch", torch.__version__)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def drive_ok(path, shape0=None):
    try:
        a = np.load(path, mmap_mode="r"); return shape0 is None or a.shape[0] == shape0
    except Exception:
        return False


## Preprocessing (skipped if cached)

In [ ]:
# Preprocessing: average repetitions, per-subject channel z-score, store as float16.
# Each split runs in a separate process to limit peak memory.
PREP_SCRIPT = r'''
import numpy as np, os, sys, time
eeg_dir, out_dir, sub, split = sys.argv[1:5]
src = f"{eeg_dir}/{sub}/preprocessed_eeg_{'training' if split == 'train' else 'test'}.npy"
t0 = time.time()
obj = np.load(src, allow_pickle=True)
d = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
arr = d["preprocessed_eeg_data"]
avg = np.empty((arr.shape[0],) + arr.shape[2:], dtype=np.float32)
for i in range(0, arr.shape[0], 500):
    avg[i:i + 500] = arr[i:i + 500].mean(axis=1)
del d, arr, obj
m = avg.mean(axis=(0, 2), keepdims=True); s = avg.std(axis=(0, 2), keepdims=True) + 1e-6
out = np.empty(avg.shape, dtype=np.float16)
for i in range(0, avg.shape[0], 500):
    out[i:i + 500] = (avg[i:i + 500] - m) / s
tmp = f"{out_dir}/{sub}_{split}_z16.tmp.npy"
np.save(tmp, out); os.replace(tmp, f"{out_dir}/{sub}_{split}_z16.npy")
print(f"  {sub}/{split}: {out.shape} saved in {time.time()-t0:.0f}s", flush=True)
'''
EEG_DIR = f"{ROOT}/Preprocessed_data_250Hz"
N_ROWS = {"train": 16540, "test": 200}
for sub in ALL_SUBJECTS:
    for split in ["train", "test"]:
        if drive_ok(f"{DCACHE}/{sub}_{split}_z16.npy", N_ROWS[split]): continue
        fname = f"preprocessed_eeg_{'training' if split == 'train' else 'test'}.npy"
        raw = f"{EEG_DIR}/{sub}/{fname}"
        if not os.path.exists(raw):
            from huggingface_hub import hf_hub_download
            hf_hub_download(repo_id="LidongYang/EEG_Image_decode", repo_type="dataset", local_dir=ROOT,
                            filename=f"Preprocessed_data_250Hz/{sub}/{fname}")
        r = subprocess.run([sys.executable, "-c", PREP_SCRIPT, EEG_DIR, DCACHE, sub, split], capture_output=True, text=True)
        print(r.stdout.strip())
        if r.returncode != 0:
            print(r.stderr[-1500:])
            raise RuntimeError(f"preparing {sub}/{split} failed (message above)")
        if os.path.exists(raw): os.remove(raw)
    print(f"{sub} done")



## Captions and features

In [ ]:
# Image captions and features for two visual targets (cached on Drive)
# B32: CLIP ViT-B/32 (as before). H14: CLIP ViT-H/14 (LAION-2B), the target used by ATM.
TARGETS = {"B32": ("ViT-B-32", "openai"), "H14": ("ViT-H-14", "laion2b_s32b_b79k")}
CAPTION_MODEL = "Salesforce/blip-image-captioning-large"
cap_path = f"{DCACHE}/captions_blip_large.json"

import tarfile
from huggingface_hub import hf_hub_download
if not os.path.exists(f"{ROOT}/images_set/image_metadata.npy"):
    tar_path = hf_hub_download(repo_id="LidongYang/EEG_Image_decode", filename="images_set.tar.gz", repo_type="dataset", local_dir=ROOT)
    with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)
obj = np.load(f"{ROOT}/images_set/image_metadata.npy", allow_pickle=True)
meta = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
train_files, test_files = list(meta["train_img_files"]), list(meta["test_img_files"])
def build_index(folder):
    idx = {}
    for dp, _, fs in os.walk(folder):
        for f in fs: idx.setdefault(f, os.path.join(dp, f))
    return idx
img_root = f"{ROOT}/images_set"
idx_train = build_index(f"{img_root}/training_images")
idx_test = build_index(f"{img_root}/test_images") if os.path.isdir(f"{img_root}/test_images") else build_index(img_root)
train_paths, test_paths = [idx_train[f] for f in train_files], [idx_test[f] for f in test_files]

def concept_names(key, n_imgs):
    c = list(meta[key])
    if len(c) != n_imgs:
        per = n_imgs // len(c); c = [x for x in c for _ in range(per)]
    clean = lambda s: s.split("_", 1)[1].replace("_", " ") if s.split("_", 1)[0].isdigit() else s.replace("_", " ")
    return [clean(x) for x in c]
train_names, test_names = concept_names("train_img_concepts", len(train_files)), concept_names("test_img_concepts", len(test_files))
uniq = sorted(set(train_names)); c2i = {c: i for i, c in enumerate(uniq)}
TRAIN_CONCEPT = np.array([c2i[c] for c in train_names])

from PIL import Image
from torch.utils.data import Dataset, DataLoader
class PathDS(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert("RGB"))

def make_captions(paths, bs=64):
    from transformers import BlipProcessor, BlipForConditionalGeneration
    proc = BlipProcessor.from_pretrained(CAPTION_MODEL)
    model = BlipForConditionalGeneration.from_pretrained(CAPTION_MODEL, torch_dtype=torch.float16).to(device).eval()
    out = []
    for i in range(0, len(paths), bs):
        ims = [Image.open(p).convert("RGB") for p in paths[i:i + bs]]
        inp = proc(images=ims, return_tensors="pt").to(device, torch.float16)
        with torch.no_grad(): ids = model.generate(**inp, max_new_tokens=30, num_beams=3)
        out += [s.strip() for s in proc.batch_decode(ids, skip_special_tokens=True)]
        if i % (bs * 40) == 0: print(f"  captions {i + len(ims)}/{len(paths)}", flush=True)
    del model; torch.cuda.empty_cache() if device == "cuda" else None
    return out

try:
    CAPS = json.load(open(cap_path)); assert len(CAPS["train"]) == len(train_paths)
    print("captions loaded")
except Exception:
    t0 = time.time()
    CAPS = {"train": make_captions(train_paths), "test": make_captions(test_paths)}
    json.dump(CAPS, open(cap_path + ".tmp", "w")); os.replace(cap_path + ".tmp", cap_path)
    print(f"captions done in {(time.time()-t0)/60:.1f} min")
for i in [0, 1, 2]: print("  example:", test_names[i], "->", CAPS["test"][i])

import open_clip
FEATS = {}
for key, (arch, pre) in TARGETS.items():
    path = f"{DCACHE}/feats3_{arch}_{pre}.pt"
    try:
        FEATS[key] = torch.load(path); print(key, "features loaded"); continue
    except Exception:
        pass
    t0 = time.time()
    model, _, preprocess = open_clip.create_model_and_transforms(arch, pretrained=pre)
    model = model.to(device).eval(); tok = open_clip.get_tokenizer(arch)
    @torch.no_grad()
    def enc_img(paths):
        out = []
        for x in DataLoader(PathDS(paths, preprocess), batch_size=128, num_workers=4):
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
                out.append(F.normalize(model.encode_image(x.to(device)).float(), dim=-1).cpu())
        return torch.cat(out).half()
    @torch.no_grad()
    def enc_txt(texts):
        out = []
        for i in range(0, len(texts), 512):
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
                out.append(F.normalize(model.encode_text(tok(texts[i:i + 512]).to(device)).float(), dim=-1).cpu())
        return torch.cat(out).half()
    cls_u = enc_txt([f"a photo of a {n}" for n in uniq])
    FEATS[key] = {"img_tr": enc_img(train_paths), "img_te": enc_img(test_paths),
                  "cap_tr": enc_txt(CAPS["train"]), "cap_te": enc_txt(CAPS["test"]),
                  "cls_tr": cls_u[torch.from_numpy(TRAIN_CONCEPT)], "cls_te": enc_txt([f"a photo of a {n}" for n in test_names]),
                  "n_concepts": len(uniq)}
    torch.save(FEATS[key], path + ".tmp"); os.replace(path + ".tmp", path)
    del model; torch.cuda.empty_cache() if device == "cuda" else None
    print(f"{key} features computed in {(time.time()-t0)/60:.1f} min")
N_TR = FEATS["B32"]["img_tr"].shape[0]


## Models and training utilities

In [ ]:
# Shared model, losses and metrics
import copy
class EEGEncoder(nn.Module):
    def __init__(self, n_ch, n_t, out_dim, k=40, drop=0.5):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, k, (1, 25)), nn.AvgPool2d((1, 51), (1, 5)), nn.BatchNorm2d(k), nn.ELU(),
            nn.Conv2d(k, k, (n_ch, 1)), nn.BatchNorm2d(k), nn.ELU(), nn.Dropout(drop))
        with torch.no_grad(): flat = self.backbone(torch.zeros(1, 1, n_ch, n_t)).numel()
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(flat, out_dim))
        self.res = nn.Sequential(nn.GELU(), nn.Linear(out_dim, out_dim), nn.Dropout(drop))
        self.norm = nn.LayerNorm(out_dim)
    def forward(self, x):
        h = self.proj(self.backbone(x.unsqueeze(1)))
        return F.normalize(self.norm(h + self.res(h)), dim=-1)

class ATMStyle(nn.Module):
    # channel-wise transformer (attention across electrodes) in front of the same conv backbone
    def __init__(self, n_ch, n_t, out_dim, d_ff=256, drop=0.25):
        super().__init__()
        self.inp = nn.Linear(n_t, n_t)
        pos = torch.zeros(n_ch, n_t); i = torch.arange(n_ch).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, n_t, 2).float() * (-math.log(10000.0) / n_t))
        pos[:, 0::2] = torch.sin(i * div); pos[:, 1::2] = torch.cos(i * div)[:, : n_t // 2]
        self.register_buffer("pos", pos)
        heads = next(h for h in [10, 5, 2, 1] if n_t % h == 0)
        self.attn = nn.TransformerEncoder(nn.TransformerEncoderLayer(n_t, heads, d_ff, drop, batch_first=True, norm_first=True), num_layers=1)
        self.conv = EEGEncoder(n_ch, n_t, out_dim)
    def forward(self, x):
        return self.conv(self.attn(self.inp(x) + self.pos))

def make_encoder(kind, out_dim):
    return (ATMStyle if kind == "atm" else EEGEncoder)(N_CH, N_T, out_dim).to(device)

@torch.no_grad()
def adabn(base, Xu, bs=512):
    # re-estimate BatchNorm statistics on unlabelled EEG of the new person
    m = copy.deepcopy(base)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d): mod.reset_running_stats(); mod.momentum = None
    m.train()
    for i in range(0, len(Xu), bs): m(Xu[i:i + bs].float())
    return m.eval()

class Adapted(nn.Module):
    # subject-specific channel mixing (identity at start) in front of a trained encoder
    def __init__(self, base, n_ch):
        super().__init__()
        self.base = base; self.mix = nn.Linear(n_ch, n_ch, bias=True)
        with torch.no_grad(): self.mix.weight.copy_(torch.eye(n_ch)); self.mix.bias.zero_()
    def forward(self, x):
        return self.base(self.mix(x.transpose(1, 2)).transpose(1, 2))

def contrastive(z, T, keys, scale):
    # symmetric InfoNCE; rows with the same key (same image or same concept) count as positives
    logits = scale * z @ T.T
    same = (keys[:, None] == keys[None, :]).float(); tgt = same / same.sum(1, keepdim=True)
    return (torch.sum(-tgt * F.log_softmax(logits, 1), 1).mean() + torch.sum(-tgt * F.log_softmax(logits.T, 1), 1).mean()) / 2

@torch.no_grad()
def embed(model, X, bs=1024):
    model.eval(); return torch.cat([model(X[i:i + bs].float()) for i in range(0, len(X), bs)])
@torch.no_grad()
def topk(z, T, ks=(1, 5)):
    sim = z @ T.T; rank = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {f"top{k}": (rank < k).float().mean().item() * 100 for k in ks}
@torch.no_grad()
def kway(z, T, ks=(2, 5, 10)):
    sim = z @ T.T; r = (sim > sim.diag().unsqueeze(1)).sum(1).double(); n = sim.shape[1] - 1
    lg = lambda a, b: torch.lgamma(a + 1) - torch.lgamma(b + 1) - torch.lgamma(a - b + 1)
    out = {}
    for k in ks:
        kk = torch.tensor(k - 1.0, dtype=torch.double, device=r.device); nn_ = torch.tensor(float(n), dtype=torch.double, device=r.device)
        p = torch.where(n - r >= k - 1, torch.exp(lg(n - r, kk) - lg(nn_, kk)), torch.zeros_like(r))
        out[f"{k}way"] = p.mean().item() * 100
    return out
def local_copy(name):
    dst = f"{LOCAL}/{name}"
    if not os.path.exists(dst): shutil.copy(f"{DCACHE}/{name}", dst + ".tmp"); os.replace(dst + ".tmp", dst)
    return dst

PER = N_TR // FEATS["B32"]["n_concepts"]
IS_VAL = (np.arange(N_TR) % PER) == (PER - 1)
VAL = torch.from_numpy(np.where(IS_VAL)[0]).to(device); TRN = torch.from_numpy(np.where(~IS_VAL)[0]).to(device)
CONCEPT = torch.from_numpy(TRAIN_CONCEPT).to(device)
_, N_CH, N_T = np.load(f"{DCACHE}/sub-01_test_z16.npy", mmap_mode="r").shape

def train_model(X, tmap, keys_img, T_img, T_txt=None, keys_txt=None, txt_w=0.0, idx=None, val=None, epochs=40, patience=10, batch=256, seed=0, log=None, enc="conv"):
    # X: (n, ch, t) float16 on GPU; tmap: row of X -> row of the target tables; keys: rows with equal keys are positives
    set_seed(seed)
    model = make_encoder(enc, T_img.shape[1])
    scale = nn.Parameter(torch.tensor(math.log(1 / 0.07), device=device))
    opt = torch.optim.AdamW(list(model.parameters()) + [scale], lr=LR, weight_decay=WD)
    best, best_state, best_ep, bad = -1, None, 0, 0
    for ep in range(epochs):
        model.train(); perm = idx[torch.randperm(len(idx), device=device)]
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]; tb = tmap[b]; s_ = scale.exp().clamp(max=100)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
                z = model(X[b].float())
            z = z.float()
            loss = contrastive(z, T_img[tb], keys_img[b], s_)
            if txt_w > 0: loss = loss + txt_w * contrastive(z, T_txt[tb], keys_txt[b], s_)
            opt.zero_grad(); loss.backward(); opt.step()
        va = val(model)
        if va > best: best, best_ep, bad, best_state = va, ep + 1, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}
        else: bad += 1
        if log: log(ep + 1, va)
        if bad >= patience: break
    model.load_state_dict(best_state)
    return model, best_ep, best


## Within-subject: visual target, encoder and text

In [ ]:
# Within-subject: visual target size and tri-modal (EEG, image, text) alignment, all 10 subjects
# text: none | caption (BLIP caption of each image) | classname ("a photo of a <concept>")
# (visual target, text, text weight, EEG encoder); ordered by priority
CONFIGS = [("H14", "none", 0.0, "conv"), ("H14", "caption", 0.1, "conv"), ("H14", "none", 0.0, "atm"), ("B32", "none", 0.0, "conv"),
           ("H14", "caption", 0.1, "atm"), ("H14", "caption", 0.3, "conv"), ("H14", "classname", 0.1, "conv")]
W_SUBJECTS = ALL_SUBJECTS
SEED = 0

def run_within(sub, tgt, text, w, enc):
    t0 = time.time(); Fe = FEATS[tgt]
    X = torch.from_numpy(np.load(local_copy(f"{sub}_train_z16.npy"))).to(device)
    Xt = torch.from_numpy(np.load(local_copy(f"{sub}_test_z16.npy"))).float().to(device)
    T_img = Fe["img_tr"].float().to(device); rows = torch.arange(N_TR, device=device)
    T_txt = {"caption": Fe["cap_tr"], "classname": Fe["cls_tr"]}.get(text, Fe["cap_tr"]).float().to(device)
    keys_txt = CONCEPT if text == "classname" else rows
    val = lambda m: topk(embed(m, X[VAL]), T_img[VAL])["top5"]
    model, best_ep, best_val = train_model(X, rows, rows, T_img, T_txt, keys_txt, w if text != "none" else 0.0, TRN, val, seed=SEED, enc=enc)
    z = embed(model, Xt); Ti, Tc, Tn = Fe["img_te"].float().to(device), Fe["cap_te"].float().to(device), Fe["cls_te"].float().to(device)
    r = {"subject": sub, "target": tgt, "text": text, "text_w": w, "encoder": enc, "best_epoch": best_ep, **topk(z, Ti), **kway(z, Ti)}
    r["eeg_to_caption_top1"] = topk(z, Tc)["top1"]; r["eeg_to_classname_top1"] = topk(z, Tn)["top1"]
    r["minutes"] = round((time.time() - t0) / 60, 2)
    return r

for tgt, text, w, enc in CONFIGS:
    for sub in W_SUBJECTS:
        path = f"{DRIVE}/w2_{tgt}_{text}_{w}_{enc}_{sub}.json"
        if os.path.exists(path): continue
        r = run_within(sub, tgt, text, w, enc); json.dump(r, open(path, "w"))
        print(f"{tgt} {text} {w} {enc} {sub}: top1 {r['top1']:.1f} top5 {r['top5']:.1f} 5way {r['5way']:.1f} | text-ret {r['eeg_to_caption_top1']:.1f} ({r['minutes']} min)", flush=True)

rows = [json.load(open(p)) for p in sorted(__import__("glob").glob(f"{DRIVE}/w2_*.json"))]
W = pd.DataFrame(rows); W.to_csv(f"{DRIVE}/results_within_v2.csv", index=False)
summary = W.groupby(["target", "encoder", "text", "text_w"])[["top1", "top5", "5way", "eeg_to_caption_top1", "eeg_to_classname_top1"]].agg(["mean", "std"]).round(1)
summary["n"] = W.groupby(["target", "encoder", "text", "text_w"]).size()
print(summary.to_string())


## Leave-one-subject-out and adaptation

In [ ]:
# Cross-subject: leave-one-subject-out with the stronger target, then calibration on the new person.
# Calibration uses k training images of the held-out subject (training concepts only; test concepts stay unseen).
import glob
X_TARGET = "H14"
X_ENCODERS = ["conv", "atm"]
X_FOLDS = ["sub-01", "sub-05", "sub-08", "sub-02", "sub-03", "sub-04", "sub-06", "sub-07", "sub-09", "sub-10"]
X_EPOCHS, X_PATIENCE, CAL_K, CAL_STEPS = 20, 5, [250, 1000, 4000], 300

def calibrate(base, Xc, Tc, method, seed=0):
    set_seed(seed)
    m = copy.deepcopy(base)
    if method == "adapter":
        m = Adapted(m, N_CH).to(device)
        for p in m.base.parameters(): p.requires_grad_(False)
        params, lr = list(m.mix.parameters()), 1e-3
    else:
        params, lr = list(m.parameters()), 1e-4
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=WD); scale = 1 / 0.07
    rows = torch.arange(len(Xc), device=device)
    for step in range(CAL_STEPS):
        m.train()
        if method == "adapter": m.base.eval()
        b = rows[torch.randperm(len(rows), device=device)[:min(256, len(rows))]]
        loss = contrastive(m(Xc[b].float()), Tc[b], b, scale)
        opt.zero_grad(); loss.backward(); opt.step()
    return m

Fe = FEATS[X_TARGET]; T_img = Fe["img_tr"].float().to(device); T_te = Fe["img_te"].float().to(device)
JOBS = [(e, s) for s in X_FOLDS[:3] for e in X_ENCODERS] + [(e, s) for s in X_FOLDS[3:] for e in X_ENCODERS]
for enc, sub in JOBS:
    path = f"{DRIVE}/x2_{X_TARGET}_{enc}_{sub}.json"
    if os.path.exists(path): print(f"{enc} {sub}: done"); continue
    t0 = time.time(); srcs = [s for s in ALL_SUBJECTS if s != sub]
    X = torch.empty((len(srcs) * N_TR, N_CH, N_T), dtype=torch.float16, device=device)
    for k, s in enumerate(srcs): X[k * N_TR:(k + 1) * N_TR] = torch.from_numpy(np.load(local_copy(f"{s}_train_z16.npy"))).to(device)
    img_of = torch.arange(len(X), device=device) % N_TR
    tr_idx = torch.where(~torch.from_numpy(np.tile(IS_VAL, len(srcs))).to(device))[0]
    vrows = [k * N_TR + VAL for k in range(len(srcs))]
    val = lambda m: float(np.mean([topk(embed(m, X[v]), T_img[VAL])["top5"] for v in vrows]))
    log = lambda ep, va: print(f"  {enc} {sub} ep {ep:2d} | val top5 {va:.2f} | {(time.time()-t0)/60:.1f} min", flush=True)
    base, best_ep, best_val = train_model(X, img_of, img_of, T_img, idx=tr_idx, val=val, epochs=X_EPOCHS, patience=X_PATIENCE, log=log, enc=enc)
    del X; gc.collect(); torch.cuda.empty_cache() if device == "cuda" else None

    Xt = torch.from_numpy(np.load(local_copy(f"{sub}_test_z16.npy"))).float().to(device)
    Xh = torch.from_numpy(np.load(local_copy(f"{sub}_train_z16.npy"))).to(device)
    z = embed(base, Xt)
    head = {"test_subject": sub, "target": X_TARGET, "encoder": enc, "best_epoch": best_ep}
    rows = [dict(head, method="none", k=0, **topk(z, T_te), **kway(z, T_te))]
    za = embed(adabn(base, Xh), Xt)   # unlabelled: all training-session EEG of the new person, no labels
    rows.append(dict(head, method="adabn (no labels)", k=0, **topk(za, T_te), **kway(za, T_te)))
    g = torch.Generator().manual_seed(0); order = torch.randperm(N_TR, generator=g).to(device)
    for k in CAL_K:
        sel = order[:k]
        for method in ["adapter", "finetune"]:
            m = calibrate(base, Xh[sel], T_img[sel], method)
            zc = embed(m, Xt)
            rows.append(dict(head, method=method, k=k, **topk(zc, T_te), **kway(zc, T_te)))
    for q in rows: q["minutes"] = round((time.time() - t0) / 60, 1)
    json.dump(rows, open(path, "w"))
    print(f"{enc} {sub}: LOSO top1 {rows[0]['top1']:.1f} top5 {rows[0]['top5']:.1f} 5way {rows[0]['5way']:.1f} | " +
          " | ".join(f"{q['method']} k={q['k']}: {q['top1']:.1f}" for q in rows[1:]), flush=True)
    del Xh, Xt; gc.collect()

rows = [q for p in sorted(glob.glob(f"{DRIVE}/x2_{X_TARGET}_*_sub-*.json")) for q in json.load(open(p))]
Xr = pd.DataFrame(rows); Xr.to_csv(f"{DRIVE}/results_loso_v2.csv", index=False)
print(Xr.groupby(["encoder", "method", "k"])[["top1", "top5", "5way"]].agg(["mean", "std"]).round(1).to_string())
print("folds per encoder:", Xr.groupby("encoder").test_subject.nunique().to_dict())


## Test-time refinement (no labels)

In [ ]:
# Test-time refinement of retrieval scores: no labels, encoder unchanged.
#   center : subtract the mean EEG embedding of the person (from unlabelled training-session EEG)
#            and the mean training-image embedding, then re-normalise
#   csls   : penalise "hub" images that are close to many EEG embeddings of this person (CSLS, k = 10)
# Run after the LOSO cell has finished. Same seeds as before, so "plain" should repeat the earlier numbers.
import glob
R_SUBS, R_K = ["sub-01", "sub-05", "sub-08"], 10
MODES = ["plain", "center", "csls", "center+csls"]
os.makedirs(f"{DRIVE}/models", exist_ok=True)
Fe = FEATS["H14"]; T_img = Fe["img_tr"].float().to(device); T_te = Fe["img_te"].float().to(device)
MU_IMG = T_img.mean(0, keepdim=True)

@torch.no_grad()
def scores(z, T, Zu, mode):
    if "center" in mode:
        mu = Zu.mean(0, keepdim=True)
        z, Zu, T = F.normalize(z - mu, dim=-1), F.normalize(Zu - mu, dim=-1), F.normalize(T - MU_IMG, dim=-1)
    sim = z @ T.T
    if "csls" in mode:
        hub = (T @ Zu.T).topk(R_K, dim=1).values.mean(1)
        sim = 2 * sim - hub[None, :]
    return sim

@torch.no_grad()
def metrics(sim, ks=(1, 5), ways=(2, 5, 10)):
    r = (sim > sim.diag().unsqueeze(1)).sum(1).double(); n = float(sim.shape[1] - 1)
    out = {f"top{k}": (r < k).float().mean().item() * 100 for k in ks}
    lg = lambda a, b: torch.lgamma(a + 1) - torch.lgamma(b + 1) - torch.lgamma(a - b + 1)
    N = torch.tensor(n, dtype=torch.double, device=r.device)
    for k in ways:
        kk = torch.tensor(k - 1.0, dtype=torch.double, device=r.device)
        p = torch.where(n - r >= k - 1, torch.exp(lg(n - r, kk) - lg(N, kk)), torch.zeros_like(r))
        out[f"{k}way"] = p.mean().item() * 100
    return out

def evaluate(m, Xt, Xu, tag):
    z, Zu = embed(m, Xt), embed(m, Xu)
    return [dict(tag, mode=md, **metrics(scores(z, T_te, Zu, md))) for md in MODES]

def show(rows):
    print("   " + " | ".join(f"{q['mode']} {q['top1']:.1f}" for q in rows), flush=True)

# within-subject
for sub in R_SUBS:
    path = f"{DRIVE}/r_within_{sub}.json"
    if os.path.exists(path): print(f"within {sub}: done"); continue
    t0 = time.time()
    X = torch.from_numpy(np.load(local_copy(f"{sub}_train_z16.npy"))).to(device)
    Xt = torch.from_numpy(np.load(local_copy(f"{sub}_test_z16.npy"))).float().to(device)
    rows_ = torch.arange(N_TR, device=device)
    val = lambda m: topk(embed(m, X[VAL]), T_img[VAL])["top5"]
    model, _, _ = train_model(X, rows_, rows_, T_img, idx=TRN, val=val, seed=0, enc="conv")
    torch.save(model.state_dict(), f"{DRIVE}/models/within_H14_conv_{sub}.pt")
    out = evaluate(model, Xt, X, {"setting": "within", "subject": sub, "method": "none", "k": 0})
    json.dump(out, open(path, "w")); print(f"within {sub} ({(time.time()-t0)/60:.1f} min)"); show(out)
    del X, Xt; gc.collect()

# leave-one-subject-out (conv encoder)
for sub in R_SUBS:
    path = f"{DRIVE}/r_loso_{sub}.json"
    if os.path.exists(path): print(f"loso {sub}: done"); continue
    t0 = time.time(); srcs = [s for s in ALL_SUBJECTS if s != sub]
    X = torch.empty((len(srcs) * N_TR, N_CH, N_T), dtype=torch.float16, device=device)
    for k, s in enumerate(srcs): X[k * N_TR:(k + 1) * N_TR] = torch.from_numpy(np.load(local_copy(f"{s}_train_z16.npy"))).to(device)
    img_of = torch.arange(len(X), device=device) % N_TR
    tr_idx = torch.where(~torch.from_numpy(np.tile(IS_VAL, len(srcs))).to(device))[0]
    vrows = [k * N_TR + VAL for k in range(len(srcs))]
    val = lambda m: float(np.mean([topk(embed(m, X[v]), T_img[VAL])["top5"] for v in vrows]))
    base, _, _ = train_model(X, img_of, img_of, T_img, idx=tr_idx, val=val, epochs=20, patience=5, enc="conv")
    torch.save(base.state_dict(), f"{DRIVE}/models/loso_H14_conv_{sub}.pt")
    del X; gc.collect(); torch.cuda.empty_cache()

    Xt = torch.from_numpy(np.load(local_copy(f"{sub}_test_z16.npy"))).float().to(device)
    Xh = torch.from_numpy(np.load(local_copy(f"{sub}_train_z16.npy"))).to(device)
    tag = {"setting": "loso", "subject": sub}
    out = evaluate(base, Xt, Xh, dict(tag, method="none", k=0))
    out += evaluate(adabn(base, Xh), Xt, Xh, dict(tag, method="adabn", k=0))
    g = torch.Generator().manual_seed(0); order = torch.randperm(N_TR, generator=g).to(device)
    for k in [1000, 4000]:
        m = calibrate(base, Xh[order[:k]], T_img[order[:k]], "finetune")
        out += evaluate(m, Xt, Xh, dict(tag, method="finetune", k=k))
    json.dump(out, open(path, "w")); print(f"loso {sub} ({(time.time()-t0)/60:.1f} min)")
    for q in range(0, len(out), len(MODES)): print(f"  {out[q]['method']} k={out[q]['k']}:", end=""); show(out[q:q + len(MODES)])
    del Xh, Xt; gc.collect()

R = pd.DataFrame([q for p in sorted(glob.glob(f"{DRIVE}/r_*_sub-*.json")) for q in json.load(open(p))])
R.to_csv(f"{DRIVE}/results_refine.csv", index=False)
print("\n==== Top-1 (200-way), mean over subjects ====")
print(R.pivot_table(index=["setting", "method", "k"], columns="mode", values="top1", aggfunc="mean")[MODES].round(1).to_string())
print("\n==== 5-way ====")
print(R.pivot_table(index=["setting", "method", "k"], columns="mode", values="5way", aggfunc="mean")[MODES].round(1).to_string())


## All subjects, and single-trial training

In [ ]:
# Run after the refinement cell (uses scores / metrics / evaluate / MODES defined there).
# Part 1: within-subject refinement for all 10 subjects (same training as before, ~1.5 min per subject).
# Part 2: ATM training recipe on sub-01, sub-05, sub-08: all single trials (4 per image, not averaged) and batch 1024.
#         Single trials are scaled with the statistics of the averaged training data, so they match the test scale.
#         Raw files are downloaded again (about 4 GB per subject) and kept only on the local disk.

# ---------- part 1 ----------
for sub in ALL_SUBJECTS:
    path = f"{DRIVE}/r_within_{sub}.json"
    if os.path.exists(path): continue
    t0 = time.time()
    X = torch.from_numpy(np.load(local_copy(f"{sub}_train_z16.npy"))).to(device)
    Xt = torch.from_numpy(np.load(local_copy(f"{sub}_test_z16.npy"))).float().to(device)
    rows_ = torch.arange(N_TR, device=device)
    val = lambda m: topk(embed(m, X[VAL]), T_img[VAL])["top5"]
    model, _, _ = train_model(X, rows_, rows_, T_img, idx=TRN, val=val, seed=0, enc="conv")
    torch.save(model.state_dict(), f"{DRIVE}/models/within_H14_conv_{sub}.pt")
    out = evaluate(model, Xt, X, {"setting": "within", "subject": sub, "method": "none", "k": 0})
    json.dump(out, open(path, "w")); print(f"within {sub} ({(time.time()-t0)/60:.1f} min)"); show(out)
    del X, Xt; gc.collect()

W10 = pd.DataFrame([q for s in ALL_SUBJECTS for q in json.load(open(f"{DRIVE}/r_within_{s}.json"))])
print("\nwithin, 10 subjects, Top-1 mean ± SD:")
print(W10.groupby("mode").top1.agg(["mean", "std"]).loc[MODES].round(1).to_string())

# ---------- part 2 ----------
S_SUBS, S_BATCH, S_EPOCHS, S_PATIENCE = ["sub-01", "sub-05", "sub-08"], 1024, 40, 10
SINGLE_SCRIPT = r'''
import numpy as np, sys, os, time
src, out = sys.argv[1:3]; t0 = time.time()
obj = np.load(src, allow_pickle=True)
d = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
arr = d["preprocessed_eeg_data"]                       # (images, repetitions, channels, time)
n, r = arr.shape[:2]
avg = arr.mean(axis=1, dtype=np.float32)
m = avg.mean(axis=(0, 2), keepdims=True); s = avg.std(axis=(0, 2), keepdims=True) + 1e-6
del avg
res = np.empty((n * r,) + arr.shape[2:], dtype=np.float16)
for i in range(0, n, 500):
    res[i * r:(i + 500) * r] = ((arr[i:i + 500].astype(np.float32) - m[:, None]) / s[:, None]).reshape(-1, *arr.shape[2:])
np.save(out, res); print(f"  {res.shape} in {time.time()-t0:.0f}s", flush=True)
'''
EEG_DIR = f"{ROOT}/Preprocessed_data_250Hz"
for sub in S_SUBS:
    path = f"{DRIVE}/s_within_{sub}.json"
    if os.path.exists(path): print(f"single-trial {sub}: done"); continue
    t0 = time.time()
    single = f"{LOCAL}/{sub}_train_single16.npy"
    if not os.path.exists(single):
        fname = "preprocessed_eeg_training.npy"; raw = f"{EEG_DIR}/{sub}/{fname}"
        if not os.path.exists(raw):
            from huggingface_hub import hf_hub_download
            hf_hub_download(repo_id="LidongYang/EEG_Image_decode", repo_type="dataset", local_dir=ROOT,
                            filename=f"Preprocessed_data_250Hz/{sub}/{fname}")
        r = subprocess.run([sys.executable, "-c", SINGLE_SCRIPT, raw, single], capture_output=True, text=True)
        print(r.stdout.strip())
        if r.returncode != 0: print(r.stderr[-1500:]); raise RuntimeError(f"single-trial prep {sub} failed")
        os.remove(raw)
    Xs = torch.from_numpy(np.load(single)).to(device)                   # (16540 * 4, 63, 250)
    REP = Xs.shape[0] // N_TR
    img_of = torch.arange(Xs.shape[0], device=device) // REP
    s_idx = torch.where(~torch.from_numpy(np.repeat(IS_VAL, REP)).to(device))[0]
    Xa = torch.from_numpy(np.load(local_copy(f"{sub}_train_z16.npy"))).to(device)       # averaged, for validation and centering
    Xt = torch.from_numpy(np.load(local_copy(f"{sub}_test_z16.npy"))).float().to(device)
    val = lambda m: topk(embed(m, Xa[VAL]), T_img[VAL])["top5"]
    model, best_ep, _ = train_model(Xs, img_of, img_of, T_img, idx=s_idx, val=val, epochs=S_EPOCHS,
                                    patience=S_PATIENCE, batch=S_BATCH, seed=0, enc="conv")
    torch.save(model.state_dict(), f"{DRIVE}/models/within_single_H14_conv_{sub}.pt")
    out = evaluate(model, Xt, Xa, {"setting": "within_single_trials", "subject": sub, "method": "none", "k": 0, "best_epoch": best_ep})
    json.dump(out, open(path, "w")); print(f"single-trial {sub} ({(time.time()-t0)/60:.1f} min, best epoch {best_ep})"); show(out)
    del Xs, Xa, Xt; gc.collect(); torch.cuda.empty_cache()

S = pd.DataFrame([q for s in S_SUBS if os.path.exists(f"{DRIVE}/s_within_{s}.json") for q in json.load(open(f"{DRIVE}/s_within_{s}.json"))])
A = W10[W10.subject.isin(S_SUBS)]
cmp = pd.concat([A.assign(recipe="averaged, batch 256"), S.assign(recipe="single trials, batch 1024")])
cmp.to_csv(f"{DRIVE}/results_single_trials.csv", index=False)
print("\nsub-01/05/08, Top-1 mean:")
print(cmp.pivot_table(index="recipe", columns="mode", values="top1", aggfunc="mean")[MODES].round(1).to_string())
